# Pipeline final — clusters de empresas × agentes sintéticos

Notebook enxuto para executar o pipeline final do projeto Aberje/Synthetic Audiences.

Fluxo:

```text
base de agentes 1000
↓
amostra estratificada padrão 200
↓
clusters de empresas
↓
agente sintético × cluster de empresa
↓
checkpoints parquet
↓
arquivo completo + agregados
```

Antes de rodar, ajuste apenas os caminhos e parâmetros da seção **1. Configuração**.

## 0. Instalação e imports

In [1]:
# Rode uma vez, se necessário:
# !pip install -U pandas numpy pydantic langchain langchain-openai tqdm pyarrow

from __future__ import annotations

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional
import math
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 220)

## 1. Configuração

In [13]:
# ============================================================
# AJUSTE AQUI
# ============================================================

PROJECT_DIR = Path(r"C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm")

# Base original com publicações/notícias.
NEWS_PARQUET = Path(
    r"C:\Users\claudio.bruno_cortex\Documents\código_python\research\database\measuraments\estudo_aberje_trends_2026.parquet"
)
# Base já clusterizada, se existir.
# Se tiver coluna de empresa, ela será usada para montar os clusters de empresa.
CLUSTERS_CSV = Path(
    r"C:\Users\claudio.bruno_cortex\Documents\código_python\research\data\alimentos_e_bebidas\cluster_temas_ambev.parquet"
)

DATA_OUTPUT_DIR = PROJECT_DIR / "data" / "output"
ABERJE_OUTPUT_DIR = PROJECT_DIR / "data" / "aberje_outputs"
REFERENCE_CONFIG_PATH = PROJECT_DIR / "data" / "reference" / "reference_config.yml"

RUN_NAME = "pipeline_final_company_clusters_x_agents_n1000_sample200_seed42"
RUN_OUTPUT_DIR = ABERJE_OUTPUT_DIR / RUN_NAME

MODEL = "gpt-4o-mini"
TEMPERATURE = 0.2

N_AGENTS_POPULATION = 1_000
N_AGENTS_SAMPLE = 200
RANDOM_STATE = 42

MAX_WORKERS = 30
CHECKPOINT_SIZE = 100
MAX_RETRIES = 2
RETRY_SLEEP = 1.5

# Para teste rápido, preencha com números pequenos.
# Para produção, deixe None.
LIMIT_AGENTS = None
LIMIT_CLUSTERS = None

FORCE_GENERATE_AGENTS = False
RUN_PIPELINE = True

DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("MODEL:", MODEL)

PROJECT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm
RUN_OUTPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\aberje_outputs\pipeline_final_company_clusters_x_agents_n1000_sample200_seed42
MODEL: gpt-4o-mini


## 2. Colunas esperadas, constructos e questionário

In [3]:
AGENT_ID_COLUMN = "id_persona"

PERSONA_COLUMNS = [
    "Persona",
]

SOCIO_DEMOGRAPHIC_COLUMNS = [
    "idade",
    "gênero",
    "escolaridade",
    "situação ocupacional",
    "estado civil",
    "condição de moradia",
    "tipo de moradia",
    "localização",
    "renda familiar mensal",
    "acesso à internet",
    "tamanho do domicílio",
    "telefone",
]

VALUES_ATTITUDES_PERCEPTIONS_COLUMNS = [
    "identificação política",
    "ideologia",
    "raça/cor",
    "geração",
    "segmento_e_bolhas",
    "tipo_de_território",
    "uf",
    "tipo_de_fontes_de_informação",
    "Religiosidade",
    "Família",
    "Honestidade",
    "Meritocracia",
    "Ideologia",
    "Discriminação",
    "Sexualidade",
    "Comportamento_social",
    "Identidade_nacional",
    "Insegurança",
    "Confiança",
]

AGENT_CONTEXT_COLUMNS = (
    [AGENT_ID_COLUMN]
    + PERSONA_COLUMNS
    + SOCIO_DEMOGRAPHIC_COLUMNS
    + VALUES_ATTITUDES_PERCEPTIONS_COLUMNS
)

ITEM_COLUMNS = [
    "VAL_01", "VAL_02", "VAL_03", "VAL_04",
    "POS_01", "POS_02", "POS_03", "POS_04", "POS_05",
    "CRE_01", "CRE_02", "CRE_03",
]

CONSTRUCT_MAP = {
    "Valencia": ["VAL_01", "VAL_02", "VAL_03", "VAL_04"],
    "Emocao_Positiva": ["POS_01", "POS_02", "POS_03", "POS_04", "POS_05"],
    "Credibilidade": ["CRE_01", "CRE_02", "CRE_03"],
}


class SurveyResponse(BaseModel):
    """Schema estruturado do questionário usado pelos agentes sintéticos."""

    # VALÊNCIA
    VAL_01: int = Field(ge=1, le=7)
    VAL_02: int = Field(ge=1, le=7)
    VAL_03: int = Field(ge=1, le=7)
    VAL_04: int = Field(ge=1, le=7)

    # EMOÇÃO POSITIVA
    POS_01: int = Field(ge=1, le=7)
    POS_02: int = Field(ge=1, le=7)
    POS_03: int = Field(ge=1, le=7)
    POS_04: int = Field(ge=1, le=7)
    POS_05: int = Field(ge=1, le=7)

    # CREDIBILIDADE
    CRE_01: int = Field(ge=1, le=7)
    CRE_02: int = Field(ge=1, le=7)
    CRE_03: int = Field(ge=1, le=7)


def build_questionnaire_text() -> str:
    """
    Mantém o mesmo questionário reduzido usado no notebook do projeto.

    Escala de todos os itens: inteiros de 1 a 7.
    """
    return """
QUESTIONÁRIO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Após ler o conteúdo acima, minha reação foi...

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

EMOÇÃO POSITIVA
Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

CREDIBILIDADE DA MENSAGEM
O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível
""".strip()

print(build_questionnaire_text())

QUESTIONÁRIO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Após ler o conteúdo acima, minha reação foi...

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

EMOÇÃO POSITIVA
Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

CREDIBILIDADE DA MENSAGEM
O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível


## 3. Funções utilitárias e prompt

In [4]:
def safe_get(row: pd.Series, column: str, default: str = "Não informado") -> str:
    """Busca uma coluna em uma linha do DataFrame, tratando ausência e valores nulos."""
    if column not in row.index:
        return default
    value = row[column]
    if pd.isna(value):
        return default
    return str(value)


def validate_columns(
    df: pd.DataFrame,
    required_columns: list[str],
    name: str,
    raise_error: bool = True,
) -> list[str]:
    """Valida se um DataFrame contém as colunas necessárias para o pipeline."""
    missing = [col for col in required_columns if col not in df.columns]
    if missing and raise_error:
        raise ValueError(f"{name} não contém todas as colunas esperadas. Colunas ausentes: {missing}")
    return missing


def format_column_block(row: pd.Series, columns: list[str]) -> str:
    """Formata um grupo de colunas como bloco textual para inserir no prompt."""
    return "\n".join(f"- {col}: {safe_get(row, col)}" for col in columns)


def first_existing_column(
    df: pd.DataFrame,
    candidates: list[str],
    required: bool = False,
) -> Optional[str]:
    """Retorna o primeiro nome de coluna existente entre candidatos possíveis."""
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise ValueError(f"Nenhuma das colunas esperadas foi encontrada: {candidates}")

    return None


def first_non_null(series: pd.Series, default: str = ""):
    """Retorna o primeiro valor não nulo de uma série."""
    valid = series.dropna()
    if valid.empty:
        return default
    return valid.iloc[0]


def mode_or_first(series: pd.Series, default: str = ""):
    """Retorna a moda; se não houver, retorna o primeiro valor válido."""
    valid = series.dropna()
    if valid.empty:
        return default
    mode = valid.mode()
    if not mode.empty:
        return mode.iloc[0]
    return valid.iloc[0]


def truncate_text(value: str, max_chars: int = 5_000) -> str:
    """Limita textos longos para controlar custo, latência e risco de estourar contexto."""
    value = "" if pd.isna(value) else str(value)
    if len(value) <= max_chars:
        return value
    return value[:max_chars].rsplit(" ", 1)[0] + " [...]"


def build_agent_context(agent_row: pd.Series) -> str:
    """Constrói o contexto completo do agente a partir da linha da base de personas."""
    persona = safe_get(agent_row, "Persona")
    socio = format_column_block(agent_row, SOCIO_DEMOGRAPHIC_COLUMNS)
    values = format_column_block(agent_row, VALUES_ATTITUDES_PERCEPTIONS_COLUMNS)

    return f"""
PERSONA

A variável Persona é a descrição narrativa principal do agente.
Ela sintetiza quem é essa pessoa, seu contexto social, seus hábitos, tensões e visão geral de mundo.
Use a Persona como primeiro guia de interpretação.

{persona}

VARIÁVEIS SOCIO-DEMOGRÁFICAS

Essas variáveis descrevem a posição social objetiva do agente:
idade, gênero, escolaridade, ocupação, estado civil, moradia, localização, renda,
acesso à internet, tamanho do domicílio e telefone.
Elas devem orientar como o agente interpreta riscos, oportunidades, empresas,
instituições, notícias e temas públicos.

{socio}

VALORES, ATITUDES E PERCEPÇÕES

Essas variáveis descrevem a visão de mundo do agente:
identificação política, ideologia, raça/cor, geração, segmento sociopolítico,
território, UF, fontes de informação e dimensões psicográficas.
Elas devem orientar a resposta do agente em termos de confiança, insegurança,
moralidade, religião, família, meritocracia, discriminação, sexualidade,
identidade nacional e comportamento social.

{values}
""".strip()


def get_agent_metadata(agent_row: pd.Series) -> dict:
    """Retorna os metadados do agente que devem acompanhar cada resposta."""
    metadata_cols = [
        "id_persona",
        "idade",
        "gênero",
        "escolaridade",
        "situação ocupacional",
        "estado civil",
        "condição de moradia",
        "tipo de moradia",
        "localização",
        "renda familiar mensal",
        "acesso à internet",
        "tamanho do domicílio",
        "telefone",
        "identificação política",
        "ideologia",
        "raça/cor",
        "geração",
        "segmento_e_bolhas",
        "tipo_de_território",
        "uf",
        "tipo_de_fontes_de_informação",
    ]
    return {col: safe_get(agent_row, col) for col in metadata_cols if col in agent_row.index}


def build_prompt(
    agent_context: str,
    news_title: str,
    news_content: str,
    news_source_type: str,
    source_familiarity_instruction: str,
    news_source_name: str = "",
) -> str:
    """Monta o prompt enviado ao modelo para cada par agente × cluster."""
    questionnaire_text = build_questionnaire_text()

    return f"""
Você é a pessoa descrita abaixo.

Sua tarefa é responder a um questionário de pesquisa como se fosse essa pessoa real.
Não responda como especialista, analista, consultor ou modelo de IA.
Responda a partir da Persona, das variáveis socio-demográficas e dos valores, atitudes e percepções.

CONTEXTO DO AGENTE

{agent_context}

NOTÍCIA / ESTÍMULO

Título:
{news_title}

Fonte:
{news_source_name}

Tipo de fonte:
{news_source_type}

Conteúdo:
{news_content}

RELAÇÃO ENTRE AGENTE E FONTE

{source_familiarity_instruction}

INSTRUÇÕES GERAIS DE RESPOSTA

- Você foi exposto ao conteúdo acima.
- Responda todos os itens do questionário.
- Use apenas números inteiros entre 1 e 7.
- Não inclua justificativas.
- Não inclua texto explicativo.
- Não inclua comentários.
- Retorne somente um JSON válido no schema solicitado.
- Se algum item parecer menos aplicável, ainda assim responda pela impressão geral do agente.

{questionnaire_text}
""".strip()

## 4. Configurar LLM estruturado

In [5]:
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY não encontrada. Configure a variável de ambiente antes de rodar."
    )

llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMPERATURE,
)

structured_llm = llm.with_structured_output(SurveyResponse)

print("LLM estruturado configurado.")

LLM estruturado configurado.


## 5. Gerar/carregar 1000 agentes e criar amostra padrão de 200

In [6]:
def import_generate_profiles():
    """
    Tenta localizar a função generate_profiles no projeto local.

    Ajuste a lista import_attempts se a função estiver em outro módulo.
    """
    project_str = str(PROJECT_DIR)
    if project_str not in sys.path:
        sys.path.append(project_str)

    import_attempts = [
        "generate_profiles",
        "src.generate_profiles",
        "src.agents.generate_profiles",
        "src.population.generate_profiles",
        "synthetic_audiences.generate_profiles",
        "synthetic_audiences.agents.generate_profiles",
    ]

    last_error = None

    for module_name in import_attempts:
        try:
            module = __import__(module_name, fromlist=["generate_profiles"])
            return getattr(module, "generate_profiles")
        except Exception as exc:
            last_error = exc

    raise ImportError(
        "Não encontrei generate_profiles automaticamente. "
        "Ajuste import_attempts dentro de import_generate_profiles() "
        "ou importe generate_profiles manualmente antes de chamar load_or_generate_agents(). "
        f"Último erro: {repr(last_error)}"
    )


def load_or_generate_agents(
    n_agents: int = N_AGENTS_POPULATION,
    seed: int = RANDOM_STATE,
    force_generate: bool = FORCE_GENERATE_AGENTS,
) -> pd.DataFrame:
    """
    Carrega a população de agentes, se existir; caso contrário, gera uma nova.

    Saídas:
    - agents_population_n1000_seed42.csv
    - agents_population_n1000_seed42_ptbr_semicolon.csv
    """
    agents_csv = DATA_OUTPUT_DIR / f"agents_population_n{n_agents}_seed{seed}.csv"
    agents_csv_ptbr = DATA_OUTPUT_DIR / f"agents_population_n{n_agents}_seed{seed}_ptbr_semicolon.csv"

    if agents_csv.exists() and not force_generate:
        df_agents = pd.read_csv(agents_csv)
        print(f"População carregada: {agents_csv} | linhas: {len(df_agents)}")
        return df_agents

    generate_profiles = import_generate_profiles()

    df_agents = generate_profiles(
        n=n_agents,
        output_csv=agents_csv,
        config_path=str(REFERENCE_CONFIG_PATH),
        seed=seed,
    )

    df_agents.to_csv(agents_csv, index=False, encoding="utf-8-sig")
    df_agents.to_csv(agents_csv_ptbr, index=False, sep=";", encoding="utf-8-sig")

    print(f"População gerada: {agents_csv} | linhas: {len(df_agents)}")
    print(f"Versão PT-BR salva: {agents_csv_ptbr}")

    return df_agents


def make_agent_sample(
    df_agents: pd.DataFrame,
    target: int = N_AGENTS_SAMPLE,
    stratify_col: str = "segmento_e_bolhas",
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Cria amostra estratificada de agentes.

    Regra:
    - preserva a distribuição proporcional de segmento_e_bolhas;
    - ajusta arredondamentos para chegar ao target;
    - nunca sorteia mais agentes do que existem em cada segmento.
    """
    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)

    if stratify_col not in df_agents.columns:
        raise ValueError(f"Coluna de estratificação ausente: {stratify_col}")

    if target > len(df_agents):
        raise ValueError(f"O target ({target}) é maior que o total disponível ({len(df_agents)}).")

    counts = df_agents[stratify_col].value_counts(dropna=False)
    proportions = counts / counts.sum()

    allocation = pd.DataFrame({
        stratify_col: counts.index,
        "available": counts.values,
        "raw_n": [target * proportions.loc[idx] for idx in counts.index],
    })

    allocation["n"] = allocation["raw_n"].apply(math.floor).astype(int)
    allocation["remainder"] = allocation["raw_n"] - allocation["n"]

    # Garante ao menos 1 agente por segmento quando o target comporta isso.
    if target >= len(allocation):
        allocation.loc[allocation["n"] == 0, "n"] = 1

    # Corrige eventual excesso gerado pela regra de mínimo 1.
    while allocation["n"].sum() > target:
        candidates = allocation[allocation["n"] > 1].sort_values("remainder", ascending=True)
        idx = candidates.index[0]
        allocation.loc[idx, "n"] -= 1

    # Distribui vagas restantes pelos maiores restos.
    while allocation["n"].sum() < target:
        candidates = allocation[allocation["n"] < allocation["available"]].sort_values(
            "remainder",
            ascending=False,
        )
        if candidates.empty:
            break
        idx = candidates.index[0]
        allocation.loc[idx, "n"] += 1

    samples = []

    for _, row in allocation.iterrows():
        segment = row[stratify_col]
        n = int(min(row["n"], row["available"]))
        df_segment = df_agents[df_agents[stratify_col].eq(segment)]

        samples.append(
            df_segment.sample(
                n=n,
                random_state=random_state,
                replace=False,
            )
        )

    df_sample = (
        pd.concat(samples, ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    sample_path = RUN_OUTPUT_DIR / f"agents_sample_n{len(df_sample)}_seed{random_state}.csv"
    df_sample.to_csv(sample_path, index=False, sep=";", encoding="utf-8-sig")

    print("Amostra salva em:", sample_path)
    print("Total amostra:", len(df_sample))
    display(df_sample[stratify_col].value_counts().to_frame("n_agentes"))

    return df_sample

## 6. Montar clusters de empresas

In [11]:
def prepare_company_clusters_from_news(
    df_news: pd.DataFrame,
    max_text_chars: int = 5_000,
) -> pd.DataFrame:
    """
    Monta uma base de clusters por empresa.

    O agrupamento padrão é:
    Empresa analisada × cluster_id

    A função aceita variações comuns de nomes de colunas para reduzir ajustes manuais.
    """
    empresa_col = first_existing_column(
        df_news,
        ["Empresa analisada", "empresa_analisada", "empresa", "Empresa"],
        required=True,
    )
    cluster_col = first_existing_column(
        df_news,
        ["cluster_id", "id_cluster", "cluster", "Cluster"],
        required=True,
    )
    tema_col = first_existing_column(
        df_news,
        ["tema", "cluster_tema", "Tema"],
        required=False,
    )
    macrotema_col = first_existing_column(
        df_news,
        ["macrotema", "macrotema_y", "cluster_macrotema", "Macrotema"],
        required=False,
    )
    resumo_col = first_existing_column(
        df_news,
        ["resumo", "cluster_resumo", "resumo_cluster", "cluster_summary"],
        required=False,
    )
    titulo_col = first_existing_column(
        df_news,
        ["titulo_da_publicacao", "titulo", "Título", "title"],
        required=False,
    )
    text_col = first_existing_column(
        df_news,
        ["text", "conteudo", "Conteúdo", "content", "texto"],
        required=False,
    )
    setor_col = first_existing_column(
        df_news,
        ["Setor", "setor"],
        required=False,
    )
    macro_setor_col = first_existing_column(
        df_news,
        ["Macro setores", "macro_setor", "Macro setor", "macrosetor"],
        required=False,
    )
    data_col = first_existing_column(
        df_news,
        ["data_da_publicacao", "Data", "data"],
        required=False,
    )

    rows = []

    for (empresa, cluster_id), group in df_news.groupby([empresa_col, cluster_col], dropna=False):
        tema = mode_or_first(group[tema_col]) if tema_col else ""
        macrotema = mode_or_first(group[macrotema_col]) if macrotema_col else ""

        if resumo_col:
            resumo_base = mode_or_first(group[resumo_col])
        else:
            parts = []

            if titulo_col:
                titulos = group[titulo_col].dropna().astype(str).head(5).tolist()
                if titulos:
                    parts.append("Títulos principais:\n- " + "\n- ".join(titulos))

            if text_col:
                textos = group[text_col].dropna().astype(str).head(3).tolist()
                if textos:
                    parts.append("Trechos de conteúdo:\n" + "\n\n".join(textos))

            resumo_base = "\n\n".join(parts)

        resumo = f"""
Empresa analisada: {empresa}
Tema do cluster: {tema}
Macrotema do cluster: {macrotema}
Número de publicações no cluster: {len(group)}

Resumo do cluster:
{truncate_text(resumo_base, max_chars=max_text_chars)}
""".strip()

        row = {
            "company_cluster_id": f"{empresa}__{cluster_id}",
            "cluster_id": str(cluster_id),
            "empresa_analisada": empresa,
            "setor": first_non_null(group[setor_col]) if setor_col else "",
            "macro_setor": first_non_null(group[macro_setor_col]) if macro_setor_col else "",
            "tema": tema,
            "macrotema": macrotema,
            "resumo": resumo,
            "n_publicacoes": len(group),
        }

        if data_col:
            datas = pd.to_datetime(group[data_col], errors="coerce")
            row["data_min"] = datas.min()
            row["data_max"] = datas.max()

        rows.append(row)

    df_clusters = pd.DataFrame(rows)

    clusters_parquet_path = RUN_OUTPUT_DIR / "company_clusters_input.parquet"
    clusters_csv_path = RUN_OUTPUT_DIR / "company_clusters_input.csv"

    df_clusters.to_parquet(clusters_parquet_path, index=False)
    df_clusters.to_csv(clusters_csv_path, index=False, sep=";", encoding="utf-8-sig")

    print("Clusters de empresas salvos em:", clusters_parquet_path)
    print("Total clusters de empresas:", len(df_clusters))

    return df_clusters


def load_company_clusters() -> pd.DataFrame:
    """
    Carrega clusters já prontos ou monta clusters por empresa a partir da base original.

    Ordem:
    1. usa CLUSTERS_CSV se existir e tiver coluna de empresa;
    2. caso contrário, usa NEWS_PARQUET.
    """
    if CLUSTERS_CSV.exists():
        df_clusters_raw = pd.read_parquet(CLUSTERS_CSV)

        empresa_col = first_existing_column(
            df_clusters_raw,
            ["Empresa analisada", "empresa_analisada", "empresa", "Empresa"],
            required=False,
        )

        if empresa_col is not None:
            print("Carregando clusters já prontos:", CLUSTERS_CSV)
            return prepare_company_clusters_from_news(df_clusters_raw)

        print(
            "CLUSTERS_CSV existe, mas não encontrei coluna de empresa. "
            "Vou tentar montar a estrutura a partir do NEWS_PARQUET."
        )

    if not NEWS_PARQUET.exists():
        raise FileNotFoundError(
            "Não encontrei CLUSTERS_CSV com empresa nem NEWS_PARQUET. "
            "Ajuste CLUSTERS_CSV ou NEWS_PARQUET na configuração."
        )

    print("Montando clusters por empresa a partir de:", NEWS_PARQUET)
    df_news = pd.read_parquet(NEWS_PARQUET)

    return prepare_company_clusters_from_news(df_news)

## 7. Rodar agentes sintéticos sobre clusters de empresas

In [8]:
def model_response_to_dict(response) -> dict:
    """Converte resposta Pydantic v1/v2 para dict."""
    if hasattr(response, "model_dump"):
        return response.model_dump()
    if hasattr(response, "dict"):
        return response.dict()
    return dict(response)


def evaluate_company_cluster_for_agent(
    agent_row: pd.Series,
    cluster_row: pd.Series,
    structured_llm,
    model: str = MODEL,
    max_retries: int = MAX_RETRIES,
    retry_sleep: float = RETRY_SLEEP,
) -> dict:
    """
    Avalia um cluster de empresa para um agente sintético.

    Cada chamada retorna:
    - respostas 1-7 do questionário;
    - metadados do agente;
    - metadados do cluster de empresa.
    """
    news_title = (
        f"Empresa: {cluster_row.get('empresa_analisada', '')} | "
        f"Tema: {cluster_row.get('tema', '')} | "
        f"Macrotema: {cluster_row.get('macrotema', '')}"
    )

    prompt = build_prompt(
        agent_context=build_agent_context(agent_row),
        news_title=news_title,
        news_content=str(cluster_row.get("resumo", "")),
        news_source_type="Cluster de empresa / resumo sintético",
        news_source_name="Resumo de cobertura midiática por empresa",
        source_familiarity_instruction=(
            "Este conteúdo representa um resumo sintético de publicações clusterizadas "
            "sobre uma empresa. Como não há veículo específico nesta etapa, não ajuste "
            "a resposta por familiaridade com fonte. Responda considerando o conteúdo "
            "do cluster, a empresa, o tema, o macrotema e o perfil do agente."
        ),
    )

    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = structured_llm.invoke(prompt)
            result = model_response_to_dict(response)
            result.update(get_agent_metadata(agent_row))
            result.update({
                "company_cluster_id": str(cluster_row.get("company_cluster_id", "")),
                "cluster_id": str(cluster_row.get("cluster_id", "")),
                "empresa_analisada": str(cluster_row.get("empresa_analisada", "")),
                "setor": str(cluster_row.get("setor", "")),
                "macro_setor": str(cluster_row.get("macro_setor", "")),
                "cluster_resumo": str(cluster_row.get("resumo", "")),
                "cluster_tema": str(cluster_row.get("tema", "")),
                "cluster_macrotema": str(cluster_row.get("macrotema", "")),
                "n_publicacoes_cluster": cluster_row.get("n_publicacoes", None),
                "data_min_cluster": str(cluster_row.get("data_min", "")),
                "data_max_cluster": str(cluster_row.get("data_max", "")),
                "news_id": str(cluster_row.get("company_cluster_id", "")),
                "news_titulo": news_title,
                "news_fonte": "Resumo de cobertura midiática por empresa",
                "news_tipo_de_fontes_de_informação": "Cluster de empresa / resumo sintético",
                "news_tema": str(cluster_row.get("tema", "")),
                "news_macro_tema": str(cluster_row.get("macrotema", "")),
                "news_empresa": str(cluster_row.get("empresa_analisada", "")),
                "news_data": str(cluster_row.get("data_max", "")),
                "news_link": "",
                "fonte_familiar": None,
                "model": model,
                "error": "",
            })
            return result

        except Exception as exc:
            last_error = repr(exc)
            if attempt < max_retries:
                time.sleep(retry_sleep * (attempt + 1))

    # Resultado de erro preserva metadados para diagnóstico e reprocessamento.
    error_result = get_agent_metadata(agent_row)
    error_result.update({
        "company_cluster_id": str(cluster_row.get("company_cluster_id", "")),
        "cluster_id": str(cluster_row.get("cluster_id", "")),
        "empresa_analisada": str(cluster_row.get("empresa_analisada", "")),
        "setor": str(cluster_row.get("setor", "")),
        "macro_setor": str(cluster_row.get("macro_setor", "")),
        "cluster_resumo": str(cluster_row.get("resumo", "")),
        "cluster_tema": str(cluster_row.get("tema", "")),
        "cluster_macrotema": str(cluster_row.get("macrotema", "")),
        "n_publicacoes_cluster": cluster_row.get("n_publicacoes", None),
        "data_min_cluster": str(cluster_row.get("data_min", "")),
        "data_max_cluster": str(cluster_row.get("data_max", "")),
        "news_id": str(cluster_row.get("company_cluster_id", "")),
        "news_titulo": str(cluster_row.get("tema", "")),
        "news_fonte": "Resumo de cobertura midiática por empresa",
        "news_tipo_de_fontes_de_informação": "Cluster de empresa / resumo sintético",
        "news_tema": str(cluster_row.get("tema", "")),
        "news_macro_tema": str(cluster_row.get("macrotema", "")),
        "news_empresa": str(cluster_row.get("empresa_analisada", "")),
        "news_data": str(cluster_row.get("data_max", "")),
        "news_link": "",
        "fonte_familiar": None,
        "model": model,
        "error": last_error,
    })
    return error_result


def iter_company_cluster_task_batches(
    df_agents_run: pd.DataFrame,
    df_clusters_run: pd.DataFrame,
    batch_size: int = CHECKPOINT_SIZE,
):
    """Gera lotes agente × cluster sem criar uma lista gigante em memória."""
    batch = []

    for cluster_idx in df_clusters_run.index:
        for agent_idx in df_agents_run.index:
            batch.append((agent_idx, cluster_idx))

            if len(batch) >= batch_size:
                yield batch
                batch = []

    if batch:
        yield batch


def run_synthetic_audience_company_clusters(
    df_agents: pd.DataFrame,
    df_clusters: pd.DataFrame,
    output_dir: str | Path = RUN_OUTPUT_DIR,
    max_workers: int = MAX_WORKERS,
    checkpoint_size: int = CHECKPOINT_SIZE,
    max_retries: int = MAX_RETRIES,
    retry_sleep: float = RETRY_SLEEP,
    limit_agents: Optional[int] = LIMIT_AGENTS,
    limit_clusters: Optional[int] = LIMIT_CLUSTERS,
) -> None:
    """
    Executa agentes sintéticos sobre clusters de empresas.

    Cada linha de saída representa 1 agente × 1 cluster de empresa.
    """
    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)
    validate_columns(
        df_clusters,
        ["company_cluster_id", "cluster_id", "empresa_analisada", "tema", "macrotema", "resumo"],
        "df_clusters",
        raise_error=True,
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df_agents_run = df_agents.copy()
    df_clusters_run = df_clusters.copy()

    if limit_agents is not None:
        df_agents_run = df_agents_run.head(limit_agents)

    if limit_clusters is not None:
        df_clusters_run = df_clusters_run.head(limit_clusters)

    total_tasks = len(df_agents_run) * len(df_clusters_run)

    print("Agentes no run:", len(df_agents_run))
    print("Clusters de empresas:", len(df_clusters_run))
    print("Total agente × cluster:", total_tasks)
    print("Max workers:", max_workers)
    print("Checkpoint size:", checkpoint_size)

    if total_tasks == 0:
        print("Nenhuma tarefa para executar.")
        return None

    checkpoint_paths = []
    processed = 0
    batch_id = 0

    for batch in iter_company_cluster_task_batches(
        df_agents_run=df_agents_run,
        df_clusters_run=df_clusters_run,
        batch_size=checkpoint_size,
    ):
        batch_id += 1
        batch_results = []

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_task = {
                executor.submit(
                    evaluate_company_cluster_for_agent,
                    df_agents_run.loc[agent_idx],
                    df_clusters_run.loc[cluster_idx],
                    structured_llm,
                    MODEL,
                    max_retries,
                    retry_sleep,
                ): (agent_idx, cluster_idx)
                for agent_idx, cluster_idx in batch
            }

            completed = as_completed(future_to_task)

            if tqdm is not None:
                completed = tqdm(
                    completed,
                    total=len(future_to_task),
                    desc=f"Batch {batch_id} | {processed}/{total_tasks}",
                )

            for future in completed:
                batch_results.append(future.result())

        df_batch = pd.DataFrame(batch_results)
        checkpoint_path = output_dir / f"company_cluster_responses_checkpoint_{batch_id:05d}.parquet"
        df_batch.to_parquet(checkpoint_path, index=False)

        checkpoint_paths.append(checkpoint_path)
        processed += len(df_batch)

        print(
            f"Checkpoint salvo: {checkpoint_path.name} | "
            f"Processados: {processed}/{total_tasks}"
        )

    manifest = pd.DataFrame({
        "checkpoint_path": [str(path) for path in checkpoint_paths],
    })

    manifest_path = output_dir / "manifest_company_cluster_checkpoints.csv"
    manifest.to_csv(manifest_path, index=False, sep=";", encoding="utf-8-sig")

    print("Manifest salvo em:", manifest_path)
    print("Run finalizado.")

## 8. Consolidar completo e gerar agregados

In [9]:
def compute_construct_scores(df_responses: pd.DataFrame) -> pd.DataFrame:
    """Calcula os constructos a partir dos itens do questionário."""
    df = df_responses.copy()

    for item_col in ITEM_COLUMNS:
        if item_col in df.columns:
            df[item_col] = pd.to_numeric(df[item_col], errors="coerce")

    for construct, items in CONSTRUCT_MAP.items():
        existing_items = [item for item in items if item in df.columns]
        if existing_items:
            df[construct] = df[existing_items].mean(axis=1)

    return df


def load_company_cluster_checkpoints(output_dir: str | Path = RUN_OUTPUT_DIR) -> pd.DataFrame:
    """Lê todos os checkpoints parquet do run e devolve um DataFrame único."""
    output_dir = Path(output_dir)
    files = sorted(output_dir.glob("company_cluster_responses_checkpoint_*.parquet"))

    if not files:
        raise FileNotFoundError(f"Nenhum checkpoint encontrado em: {output_dir}")

    print("Checkpoints encontrados:", len(files))
    df = pd.concat([pd.read_parquet(file) for file in files], ignore_index=True)
    print("Shape consolidado:", df.shape)

    return df


def count_errors(series: pd.Series) -> int:
    """Conta respostas com erro registrado."""
    return int((series.fillna("").astype(str).str.len() > 0).sum())


def aggregate_company_cluster_scores(df_scores: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Gera tabelas agregadas para análise final.

    Retorna:
    - agg_cluster_segment: empresa × cluster × segmento;
    - agg_company_segment: empresa × segmento;
    - agg_segment: segmento;
    - agg_company_cluster: empresa × cluster geral.
    """
    df_scores = df_scores.copy()

    construct_cols = [col for col in CONSTRUCT_MAP.keys() if col in df_scores.columns]
    item_cols = [col for col in ITEM_COLUMNS if col in df_scores.columns]
    base_metrics = item_cols + construct_cols

    for col in base_metrics:
        df_scores[col] = pd.to_numeric(df_scores[col], errors="coerce")

    agg_cluster_segment = (
        df_scores
        .groupby(
            [
                "empresa_analisada",
                "setor",
                "macro_setor",
                "company_cluster_id",
                "cluster_id",
                "cluster_tema",
                "cluster_macrotema",
                "segmento_e_bolhas",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_publicacoes_cluster=("n_publicacoes_cluster", "max"),
            **{f"{col}_mean": (col, "mean") for col in base_metrics},
            **{f"{col}_std": (col, "std") for col in base_metrics},
            errors=("error", count_errors),
        )
    )

    agg_company_segment = (
        df_scores
        .groupby(
            ["empresa_analisada", "setor", "macro_setor", "segmento_e_bolhas"],
            dropna=False,
            as_index=False,
        )
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_clusters=("company_cluster_id", "nunique"),
            n_publicacoes=("n_publicacoes_cluster", "sum"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    agg_segment = (
        df_scores
        .groupby(["segmento_e_bolhas"], dropna=False, as_index=False)
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_empresas=("empresa_analisada", "nunique"),
            n_clusters=("company_cluster_id", "nunique"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    agg_company_cluster = (
        df_scores
        .groupby(
            [
                "empresa_analisada",
                "setor",
                "macro_setor",
                "company_cluster_id",
                "cluster_id",
                "cluster_tema",
                "cluster_macrotema",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_segmentos=("segmento_e_bolhas", "nunique"),
            n_publicacoes_cluster=("n_publicacoes_cluster", "max"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    for df_agg in [agg_cluster_segment, agg_company_segment, agg_segment, agg_company_cluster]:
        numeric_cols = df_agg.select_dtypes(include="number").columns
        df_agg[numeric_cols] = df_agg[numeric_cols].round(3)

    return {
        "agg_cluster_segment": agg_cluster_segment,
        "agg_company_segment": agg_company_segment,
        "agg_segment": agg_segment,
        "agg_company_cluster": agg_company_cluster,
    }


def save_complete_and_aggregated_outputs(
    output_dir: str | Path = RUN_OUTPUT_DIR,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Consolida os checkpoints, calcula constructos e salva outputs finais.

    Arquivos principais:
    - synthetic_company_clusters_complete.parquet
    - synthetic_company_clusters_complete.csv
    - aggregations/*.parquet
    - aggregations/*.csv
    """
    output_dir = Path(output_dir)
    agg_dir = output_dir / "aggregations"
    agg_dir.mkdir(parents=True, exist_ok=True)

    df_complete = load_company_cluster_checkpoints(output_dir)
    df_complete = compute_construct_scores(df_complete)

    complete_parquet = output_dir / "synthetic_company_clusters_complete.parquet"
    complete_csv = output_dir / "synthetic_company_clusters_complete.csv"

    df_complete.to_parquet(complete_parquet, index=False)
    df_complete.to_csv(complete_csv, index=False, sep=";", encoding="utf-8-sig")

    print("Arquivo completo salvo em:", complete_parquet)
    print("Arquivo completo CSV salvo em:", complete_csv)

    aggregations = aggregate_company_cluster_scores(df_complete)

    for name, df_agg in aggregations.items():
        parquet_path = agg_dir / f"{name}.parquet"
        csv_path = agg_dir / f"{name}.csv"

        df_agg.to_parquet(parquet_path, index=False)
        df_agg.to_csv(csv_path, index=False, sep=";", encoding="utf-8-sig")

        print(f"Agregado salvo: {parquet_path}")

    return df_complete, aggregations

## 9. Execução final

In [14]:
if RUN_PIPELINE:
    # 1. Carrega ou gera a população de 1000 agentes.
    df_agents_population = load_or_generate_agents(
        n_agents=N_AGENTS_POPULATION,
        seed=RANDOM_STATE,
        force_generate=FORCE_GENERATE_AGENTS,
    )

    # 2. Cria a amostra padrão de 200 agentes.
    df_agents_sample = make_agent_sample(
        df_agents=df_agents_population,
        target=N_AGENTS_SAMPLE,
        stratify_col="segmento_e_bolhas",
        random_state=RANDOM_STATE,
    )

    # 3. Carrega ou monta os clusters de empresas.
    df_company_clusters = load_company_clusters()

    display(df_company_clusters.head())

    # 4. Roda agente sintético × cluster de empresa.
    run_synthetic_audience_company_clusters(
        df_agents=df_agents_sample,
        df_clusters=df_company_clusters,
        output_dir=RUN_OUTPUT_DIR,
        max_workers=MAX_WORKERS,
        checkpoint_size=CHECKPOINT_SIZE,
        max_retries=MAX_RETRIES,
        retry_sleep=RETRY_SLEEP,
        limit_agents=LIMIT_AGENTS,
        limit_clusters=LIMIT_CLUSTERS,
    )

    # 5. Consolida completo e salva agregados.
    df_company_cluster_complete, company_cluster_aggs = save_complete_and_aggregated_outputs(
        output_dir=RUN_OUTPUT_DIR,
    )

    print("Pipeline concluído.")
    print("Completo:", RUN_OUTPUT_DIR / "synthetic_company_clusters_complete.parquet")
    print("Agregados:", RUN_OUTPUT_DIR / "aggregations")
else:
    print("RUN_PIPELINE está False. Altere para True para executar.")

População carregada: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\agents_population_n1000_seed42.csv | linhas: 1000
Amostra salva em: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\aberje_outputs\pipeline_final_company_clusters_x_agents_n1000_sample200_seed42\agents_sample_n200_seed42.csv
Total amostra: 200


,n_agentes
segmento_e_bolhas,
Conservadores cristãos,52
Dependentes do Estado / Classe D e E,42
Progressistas,24
Agro,23
Militantes de esquerda,19
Empresários,14
Empreendedores individuais,10
Liberais sociais,9
Extrema direita,7


CLUSTERS_CSV existe, mas não encontrei coluna de empresa. Vou tentar montar a estrutura a partir do NEWS_PARQUET.
Montando clusters por empresa a partir de: C:\Users\claudio.bruno_cortex\Documents\código_python\research\database\measuraments\estudo_aberje_trends_2026.parquet


KeyboardInterrupt: 

## 10. Conferência dos arquivos gerados

In [ ]:
print("Pasta do run:", RUN_OUTPUT_DIR)

for path in sorted(RUN_OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path)